In [ ]:
import pandas as pd

matches_sql = pd.read_csv("../Data/cleaned/WorldCupMatches_featured.csv")

df_sql = matches_sql.copy()

df_sql['Datetime'] = pd.to_datetime(df_sql['Datetime'], errors='coerce')
df_sql = df_sql.dropna(subset=['Datetime'])

numeric_cols = [
    'Year',
    'Attendance',
    'HomeTeamGoals',
    'AwayTeamGoals',
    'GoalDifference'
]


for col in numeric_cols:
    df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')

df_sql = df_sql.dropna(subset=numeric_cols)

df_sql[numeric_cols] = df_sql[numeric_cols].astype(int)

insert_columns = [
    'Year',
    'Datetime',
    'Stage',
    'Stadium',
    'City',
    'HomeTeamName',
    'AwayTeamName',
    'Winconditions',
    'Attendance',
    'HomeTeamGoals',
    'AwayTeamGoals',
    'GoalDifference',
    'Winner',
    'ResultCategory',
    'HighScoring',
    'KnockoutMatch'
]

df_insert = df_sql[insert_columns]

print(df_insert.columns.tolist())





In [ ]:
import pyodbc

connection = pyodbc.connect(
     'DRIVER={SQL Server};'
    'SERVER=DESKTOP-JMUSLO6;'
    'DATABASE=FIFAWorldCupAnalytics;'
    'Trusted_Connection=yes;',
    autocommit=False
)

cursor = connection.cursor()

In [ ]:
cursor.fast_executemany = True

In [ ]:
query = """
    INSERT INTO WorldCupMatches_Fact (
        Year,
        Datetime,
        Stage,
        Stadium,
        City,
        HomeTeamName,
        AwayTeamName,
        Winconditions,
        Attendance,
        HomeTeamGoals,
        AwayTeamGoals,
        GoalDifference,
        Winner,
        ResultCategory,
        HighScoringMatch,
        KnockoutMatch
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

In [ ]:
data = df_insert.values.tolist()

In [ ]:
batch_size = 1000

try:
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        cursor.executemany(query, batch)
        connection.commit()

    print(f"Inserted batch {i // batch_size + 1} of {len(data) // batch_size + 1}")

except Exception as e:
    print(f"An error occurred: {e}")
    connection.rollback()
finally:
    cursor.close()
    connection.close()
